# Pipeline GOOSE IDS no `tofino-model` — notebook único

Substitui o fluxo de 6 terminais: **conversão → validação → compilação → tofino-model → bf_switchd → carga das tabelas → injeção + `hits()` → relatório**.

**Como usar**

1. Execute a Seção 0a (dependências/helpers) e a Seção 0b (nome da sessão + conjunto de regras). Clique em **Confirmar sessão**.
2. `Run All` a partir da Seção 1. A senha do `sudo` é pedida uma única vez (Seção 1) e nunca aparece em log/relatório.
3. A Seção 7 injeta tráfego N vezes **com a mesma sessão `bfrt_python` aberta** e calcula os deltas de `hits()`.
4. A Seção 8 gera `relatorio.md`/`relatorio.html` no diretório da sessão; a Seção 9 encerra tudo.

Cada célula é reexecutável e termina com um status ✅/⚠️/❌. Parâmetros editáveis ficam em MAIÚSCULAS no topo de cada seção.

> Ambiente fixo: `open-p4studio` em `~/open-p4studio`, ambiente via `~/setup-open-p4studio.bash`, conversor em `/home/lucas/Documentos/Mestrado/Goose/Conversor/`.

## Seção 0a — Dependências, coletor de relatório e helpers

Não instala nada automaticamente: apenas informa o `pip install` necessário. Define `REPORT`, `registrar()`, `sh()` e utilitários usados por **todas** as células.

In [1]:
# ----------------------------- PARÂMETROS ----------------------------
CONVERSOR_DIR   = "/home/lucas/Documentos/Mestrado/Goose/Conversor"   # único caminho fixo do notebook
SDE_HOME_HINT   = "~/open-p4studio"
SETUP_SCRIPT    = "~/setup-open-p4studio.bash"
PROG            = "goose_ids"
TRUNCAR_RELATORIO_CHARS = 6000        # saídas maiores são truncadas no corpo do relatório (íntegra fica no log)
# ---------------------------------------------------------------------

import os, sys, re, io, json, time, shlex, signal, socket, hashlib, atexit, subprocess, platform, importlib
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field, asdict

# --- dependências -----------------------------------------------------
_DEPS = {"ipywidgets": "ipywidgets", "pexpect": "pexpect", "pandas": "pandas", "markdown": "markdown (opcional)"}
_faltando = []
for _mod, _pip in _DEPS.items():
    try:
        importlib.import_module(_mod)
    except ImportError:
        _faltando.append(_pip)
if _faltando:
    print("⚠️  Dependências ausentes:", ", ".join(_faltando))
    print("    pip install " + " ".join(m.split()[0] for m in _faltando))
    if any(m.split()[0] in ("pexpect", "pandas") for m in _faltando):
        print("    pexpect e pandas são OBRIGATÓRIOS (Seções 6–8).")
else:
    print("✅ dependências: ipywidgets, pexpect, pandas, markdown")

import pandas as pd
try:
    import pexpect
except ImportError:
    pexpect = None

CONVERSOR = Path(CONVERSOR_DIR).expanduser().resolve()
SDE_HOME  = Path(SDE_HOME_HINT).expanduser().resolve()
BUILD_DIR = CONVERSOR / "build"
P4_DST    = SDE_HOME / "pkgsrc/p4-examples/p4_16_programs" / PROG
assert CONVERSOR.is_dir(), f"Diretório do conversor não existe: {CONVERSOR}"

# --- segredos (nunca impressos) --------------------------------------
_SECRETS = {"sudo": None}
def _redigir(texto):
    """Remove a senha do sudo de qualquer texto antes de registrar/exibir."""
    if not texto:
        return texto or ""
    pw = _SECRETS.get("sudo")
    return texto.replace(pw, "«REDIGIDO»") if pw else texto

_ANSI = re.compile(r"\x1b\[[0-9;?]*[ -/]*[@-~]|\x1b\][^\x07]*\x07|\r")
def limpar_ansi(s):
    return _ANSI.sub("", s or "")

# --- estado da sessão ------------------------------------------------
@dataclass
class Sessao:
    nome: str
    regras: Path
    inicio: datetime
    slug: str
    dir: Path
    status: dict = field(default_factory=dict)       # secao -> OK / AVISO / FALHA
    duracao: dict = field(default_factory=dict)      # secao -> segundos
    pids: dict = field(default_factory=dict)         # 'model' / 'switchd' -> pgid
    dados: dict = field(default_factory=dict)        # métricas parseadas (conversão, validação, ...)
    def __repr__(self):
        return f"Sessao(nome={self.nome!r}, regras={self.regras.name}, dir={self.dir})"

SESSION = None
FORCAR_CONTINUACAO = False   # sobrescreve o bloqueio por FALHA (use conscientemente)

def exigir_sessao():
    if SESSION is None:
        raise RuntimeError("❌ Sessão não confirmada. Execute a Seção 0b e clique em 'Confirmar sessão'.")
    return SESSION

def exigir_etapas(*secoes):
    """Aborta se alguma seção anterior estiver FALHA (ou não executada), salvo FORCAR_CONTINUACAO."""
    s = exigir_sessao()
    for sec in secoes:
        st = s.status.get(sec)
        if st == "FALHA" and not FORCAR_CONTINUACAO:
            raise RuntimeError(f"❌ Seção {sec} está em FALHA. Corrija ou defina FORCAR_CONTINUACAO = True.")
        if st is None and not FORCAR_CONTINUACAO:
            raise RuntimeError(f"❌ Seção {sec} ainda não foi executada com sucesso nesta sessão.")

def marcar(secao, status, inicio=None):
    s = exigir_sessao()
    # nunca rebaixar FALHA -> OK silenciosamente dentro da mesma execução; a célula decide
    s.status[secao] = status
    if inicio is not None:
        s.duracao[secao] = s.duracao.get(secao, 0.0) + (time.time() - inicio)
    icone = {"OK": "✅", "AVISO": "⚠️", "FALHA": "❌"}[status]
    print(f"\n{icone} Seção {secao}: {status}")

# --- coletor de relatório --------------------------------------------
REPORT = []   # lista de dicts em ordem de execução

def registrar(secao, titulo, comando, stdout="", stderr="", returncode=None,
              inicio=None, fim=None, status="OK", tipo="shell", extra=None):
    ent = dict(secao=str(secao), titulo=titulo, comando=_redigir(str(comando or "")),
               stdout=_redigir(limpar_ansi(stdout)), stderr=_redigir(limpar_ansi(stderr)),
               returncode=returncode, inicio=inicio, fim=fim, status=status, tipo=tipo,
               extra=extra or {})
    REPORT.append(ent)
    return ent

def registrar_tabela(secao, titulo, df, status="OK"):
    txt = df.to_string() if isinstance(df, pd.DataFrame) else str(df)
    return registrar(secao, titulo, None, stdout=txt, tipo="tabela", status=status,
                     extra={"markdown": df.to_markdown(index=False) if isinstance(df, pd.DataFrame) else txt})

def registrar_alerta(secao, titulo, mensagem, status="AVISO"):
    print(("⚠️ " if status == "AVISO" else "❌ ") + mensagem)
    return registrar(secao, titulo, None, stdout=mensagem, tipo="alerta", status=status)

class ShResult:
    def __init__(self, cmd, rc, out, err, dt):
        self.cmd, self.returncode, self.stdout, self.stderr, self.dt = cmd, rc, out, err, dt
    @property
    def ok(self): return self.returncode == 0

def sh(cmd, secao, titulo, sudo=False, timeout=600, cwd=None, mostrar=True, env=None, input_extra=None):
    """Executa um comando via bash -c, registra no REPORT e devolve ShResult. Nunca bloqueia sem timeout."""
    if sudo:
        if not _SECRETS["sudo"]:
            raise RuntimeError("❌ Senha do sudo não configurada. Execute a Seção 1.")
        argv = ["sudo", "-S", "-p", "", "-E", "bash", "-c", cmd]
        entrada = _SECRETS["sudo"] + "\n" + (input_extra or "")
    else:
        argv = ["bash", "-c", cmd]
        entrada = input_extra
    ini = datetime.now(); t0 = time.time()
    try:
        p = subprocess.run(argv, input=entrada, capture_output=True, text=True, timeout=timeout,
                           cwd=str(cwd) if cwd else None, env=env or os.environ.copy())
        rc, out, err = p.returncode, p.stdout, p.stderr
    except subprocess.TimeoutExpired as e:
        rc, out, err = 124, (e.stdout or b"").decode(errors="replace") if isinstance(e.stdout, bytes) else (e.stdout or ""), \
                       f"TIMEOUT após {timeout}s"
    dt = time.time() - t0
    out, err = _redigir(out), _redigir(err)
    st = "OK" if rc == 0 else "FALHA"
    registrar(secao, titulo, ("sudo " if sudo else "") + cmd, out, err, rc, ini, datetime.now(), st)
    if mostrar:
        print(f"$ {'sudo ' if sudo else ''}{cmd}")
        if out.strip(): print(out.rstrip())
        if err.strip(): print("[stderr]", err.rstrip())
        print(f"[rc={rc}, {dt:.1f}s]")
    return ShResult(cmd, rc, out, err, dt)

def salvar_saida(nome, texto):
    """Grava uma saída íntegra no diretório da sessão."""
    s = exigir_sessao()
    p = s.dir / nome
    p.write_text(_redigir(texto or ""), encoding="utf-8")
    return p

def sha256_arquivo(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 20), b""):
            h.update(bloco)
    return h.hexdigest()

print("✅ helpers carregados. CONVERSOR =", CONVERSOR)

✅ dependências: ipywidgets, pexpect, pandas, markdown
✅ helpers carregados. CONVERSOR = /home/lucas/Documentos/Mestrado/Goose/Conversor


## Seção 0b — Configuração da sessão

Escolha o **nome da sessão** e o **conjunto de regras** (`rules_*.py` em `Conversor/`). Nada fora desta célula fixa um caminho de regras. Reexecute esta célula para iniciar outra sessão sem reiniciar o kernel.

In [2]:
# ----------------------------- PARÂMETROS ----------------------------
NOME_SESSAO_PADRAO = "Seção de teste do modelo de regras gerado pela: gpt-oss"
# ---------------------------------------------------------------------
import unicodedata

def _natural_key(p):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", p.name)]

def listar_regras():
    linhas = []
    for p in sorted(CONVERSOR.glob("rules_*.py"), key=_natural_key):
        try:
            n_rules = len(re.findall(r"^\s*def\s+rule_\w+\s*\(", p.read_text(encoding="utf-8", errors="replace"), re.M))
        except Exception:
            n_rules = -1
        mtime = datetime.fromtimestamp(p.stat().st_mtime).strftime("%Y-%m-%d %H:%M")
        linhas.append((p, mtime, n_rules))
    return linhas

def slugify(nome):
    s = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode()
    s = re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")
    return s[:60] or "sessao"

def confirmar_sessao(nome, regras_path):
    global SESSION, REPORT
    regras_path = Path(regras_path).resolve()
    assert regras_path.is_file(), f"arquivo de regras não existe: {regras_path}"
    ini = datetime.now()
    slug = slugify(nome)
    sdir = CONVERSOR / "sessoes" / f"{ini.strftime('%Y%m%d_%H%M%S')}_{slug}"
    sdir.mkdir(parents=True, exist_ok=True)
    SESSION = Sessao(nome=nome, regras=regras_path, inicio=ini, slug=slug, dir=sdir)
    REPORT = []
    registrar("0", "Sessão confirmada", None,
              stdout=f"nome={nome}\nregras={regras_path}\nsha256={sha256_arquivo(regras_path)}\ndir={sdir}",
              tipo="info")
    SESSION.status["0"] = "OK"
    print(f"✅ SESSION confirmada\n   nome ....: {nome}\n   regras ..: {regras_path.name}\n   dir .....: {sdir}")
    return SESSION

_opcoes = listar_regras()
if not _opcoes:
    raise RuntimeError(f"❌ Nenhum rules_*.py encontrado em {CONVERSOR}")

try:
    import ipywidgets as W
    from IPython.display import display
    _w_nome = W.Text(value=NOME_SESSAO_PADRAO, description="Sessão:", layout=W.Layout(width="90%"))
    _w_regras = W.Dropdown(
        options=[(f"{p.name}   [{mt}]   {n} regras", str(p)) for p, mt, n in _opcoes],
        description="Regras:", layout=W.Layout(width="90%"))
    _w_btn = W.Button(description="Confirmar sessão", button_style="success")
    _w_out = W.Output()
    def _on_click(_):
        with _w_out:
            _w_out.clear_output()
            confirmar_sessao(_w_nome.value.strip() or NOME_SESSAO_PADRAO, _w_regras.value)
    _w_btn.on_click(_on_click)
    display(W.VBox([_w_nome, _w_regras, _w_btn, _w_out]))
    print("Preencha e clique em 'Confirmar sessão' antes de seguir.")
except ImportError:
    print("ipywidgets indisponível — modo texto.")
    for i, (p, mt, n) in enumerate(_opcoes, 1):
        print(f"  {i:2d}  {p.name:<20} [{mt}]  {n} regras")
    _nome = input(f"Nome da sessão [{NOME_SESSAO_PADRAO}]: ").strip() or NOME_SESSAO_PADRAO
    _idx = int(input("Número do conjunto de regras: ").strip() or "1")
    confirmar_sessao(_nome, _opcoes[_idx - 1][0])

Preencha e clique em 'Confirmar sessão' antes de seguir.


## Seção 1 — Ambiente

Carrega `~/setup-open-p4studio.bash` **no processo do kernel** (um `!source` não persiste entre células), pede a senha do `sudo` uma única vez e cria as veths de forma idempotente.

In [3]:
# ----------------------------- PARÂMETROS ----------------------------
N_VETHS = 128
# ---------------------------------------------------------------------
exigir_sessao(); _t0 = time.time(); _sec = "1"; _st = "OK"

# 1) ambiente do SDE no kernel
_setup = Path(SETUP_SCRIPT).expanduser()
if not _setup.is_file():
    marcar(_sec, "FALHA", _t0); raise RuntimeError(f"❌ script de ambiente não encontrado: {_setup}")
_p = subprocess.run(["bash", "-c", f"source {shlex.quote(str(_setup))} >/dev/null 2>&1; env -0"],
                    capture_output=True, timeout=60)
_novo_env = {}
for _par in _p.stdout.split(b"\0"):
    if b"=" in _par:
        k, _, v = _par.partition(b"=")
        _novo_env[k.decode(errors="replace")] = v.decode(errors="replace")
os.environ.update(_novo_env)
SDE, SDE_INSTALL = os.environ.get("SDE", ""), os.environ.get("SDE_INSTALL", "")
_info = f"SDE.........: {SDE}\nSDE_INSTALL.: {SDE_INSTALL}\nPython .....: {sys.version.split()[0]}\nhost .......: {platform.node()}"
print(_info)
registrar(_sec, "Ambiente do SDE carregado no kernel", f"source {_setup} && env -0", stdout=_info, tipo="info")
if not SDE or not SDE_INSTALL:
    marcar(_sec, "FALHA", _t0); raise RuntimeError("❌ $SDE vazio — o script de ambiente não define SDE/SDE_INSTALL.")
SDE_HOME = Path(SDE).resolve(); P4_DST = SDE_HOME / "pkgsrc/p4-examples/p4_16_programs" / PROG
TOFINOPD = Path(SDE_INSTALL) / "share/tofinopd" / PROG
CONF     = Path(SDE_INSTALL) / "share/p4/targets/tofino" / f"{PROG}.conf"

# 2) sudo: senha uma única vez, validada com sudo -S -v
if not _SECRETS["sudo"]:
    import getpass
    for _tent in range(3):
        _pw = getpass.getpass("Senha do sudo (pedida uma única vez): ")
        _v = subprocess.run(["sudo", "-S", "-p", "", "-k", "-v"], input=_pw + "\n",
                            capture_output=True, text=True, timeout=30)
        if _v.returncode == 0:
            _SECRETS["sudo"] = _pw; del _pw
            print("✅ sudo validado"); break
        print("senha incorreta")
    else:
        marcar(_sec, "FALHA", _t0); raise RuntimeError("❌ não foi possível validar o sudo.")
else:
    print("✅ sudo já validado nesta sessão do kernel")
registrar(_sec, "sudo validado", "sudo -S -v", stdout="ok", tipo="info")

# 3) veths idempotentes
_r = sh("ip link show veth0 >/dev/null 2>&1 && echo existe || echo ausente", _sec, "veth0 existe?", mostrar=False)
if "ausente" in _r.stdout:
    _r2 = sh(f"{shlex.quote(SDE_INSTALL)}/bin/veth_setup.sh {N_VETHS}", _sec, f"veth_setup.sh {N_VETHS}", sudo=True, timeout=300)
    if not _r2.ok:
        _st = "FALHA"
else:
    print("veths já existem — veth_setup.sh não executado")
_r3 = sh("ip -br link show | grep veth | head", _sec, "veths (head)")
if "veth" not in _r3.stdout:
    registrar_alerta(_sec, "veths ausentes", "Nenhuma veth encontrada após veth_setup.sh", "FALHA"); _st = "FALHA"

# 4) versões
sh("p4c --version 2>&1 | head -3", _sec, "p4c --version")
sh("which tcpreplay tcpdump || true", _sec, "tcpreplay/tcpdump")
marcar(_sec, _st, _t0)

SDE.........: /home/lucas/open-p4studio
SDE_INSTALL.: /home/lucas/open-p4studio/install
Python .....: 3.10.12
host .......: stuxnet


Senha do sudo (pedida uma única vez):  ········


✅ sudo validado
$ sudo /home/lucas/open-p4studio/install/bin/veth_setup.sh 128
No of Veths is 128
Adding CPU veth
Added: veth0,veth1
Added: veth2,veth3
Added: veth4,veth5
Added: veth6,veth7
Added: veth8,veth9
Added: veth10,veth11
Added: veth12,veth13
Added: veth14,veth15
Added: veth16,veth17
Added: veth18,veth19
Added: veth20,veth21
Added: veth22,veth23
Added: veth24,veth25
Added: veth26,veth27
Added: veth28,veth29
Added: veth30,veth31
Added: veth32,veth33
Added: veth34,veth35
Added: veth36,veth37
Added: veth38,veth39
Added: veth40,veth41
Added: veth42,veth43
Added: veth44,veth45
Added: veth46,veth47
Added: veth48,veth49
Added: veth50,veth51
Added: veth52,veth53
Added: veth54,veth55
Added: veth56,veth57
Added: veth58,veth59
Added: veth60,veth61
Added: veth62,veth63
Added: veth64,veth65
Added: veth66,veth67
Added: veth68,veth69
Added: veth70,veth71
Added: veth72,veth73
Added: veth74,veth75
Added: veth76,veth77
Added: veth78,veth79
Added: veth80,veth81
Added: veth82,veth83
Added: veth84,

## Seção 2 — Conversão (`rules2p4.py --report`)

Gera `build/goose_ids.p4` e `build/setup_rules.py`, parseia o relatório e aplica os alertas de expansão ternária (aviso > 4× regras; erro > 2048 = `size` da tabela `detect`).

In [9]:
# ----------------------------- PARÂMETROS ----------------------------
LIMITE_TERNARIAS_DETECT = 2048   # size da tabela detect no P4 gerado
FATOR_AVISO_TERNARIAS   = 4      # aviso se entradas > FATOR × regras
# ---------------------------------------------------------------------
exigir_etapas("1"); _t0 = time.time(); _sec = "2"; _st = "OK"; S = SESSION

_r = sh(f"python3 rules2p4.py {shlex.quote(str(S.regras))} -o build --prog {PROG} --report",
        _sec, "rules2p4.py --report", cwd=CONVERSOR, timeout=600)
salvar_saida("02_conversao.txt", _r.stdout + ("\n[stderr]\n" + _r.stderr if _r.stderr.strip() else ""))

if not _r.ok:
    _dica = ""
    if "não mapeados" in _r.stderr + _r.stdout or "nao mapeados" in _r.stderr + _r.stdout:
        _dica = "→ regra usa campo novo: adicione a FIELDS em field_model.py (largura, escala, sinal)."
    elif "disjun" in _r.stderr + _r.stdout:
        _dica = "→ regra usa 'or': separe em duas funções rule_* (o match ternário faz a união)."
    elif "comparação precisa" in _r.stderr + _r.stdout:
        _dica = "→ comparação entre dois campos/aritmética: reescreva a regra ou pré-compute o valor."
    registrar_alerta(_sec, "Conversor falhou", f"rules2p4.py rc={_r.returncode}. {_dica}", "FALHA")
    marcar(_sec, "FALHA", _t0); raise RuntimeError("❌ conversão falhou — veja a saída acima e 02_conversao.txt")

# --- parse das métricas ---
def _num(rotulo, txt):
    m = re.search(rotulo + r"\s*\.*\s*(\d+)", txt)
    return int(m.group(1)) if m else None
_conv = {
    "regras_lidas":      _num(r"regras lidas", _r.stdout),
    "campos_ativos":     _num(r"campos ativos", _r.stdout),
    "entradas_ternarias":_num(r"entradas tern[aá]rias", _r.stdout),
    "classes_ataque":    _num(r"classes de ataque", _r.stdout),
}
_faixas = re.findall(r"^\s*(\w+)\s+(\d+)\s+faixas?,\s*(\d+)\s+bits?", _r.stdout, re.M)
_ataques = re.findall(r"^\s*(\d+)\s+([A-Za-z_]\w*)\s*$", _r.stdout.split("--- ataques ---")[-1], re.M) if "--- ataques ---" in _r.stdout else []
S.dados["conversao"] = dict(_conv, faixas={f: (int(n), int(b)) for f, n, b in _faixas},
                            ataques={int(i): a for i, a in _ataques})
df_conv = pd.DataFrame([_conv]); display(df_conv); registrar_tabela(_sec, "Métricas da conversão", df_conv)
if _faixas:
    df_faixas = pd.DataFrame(_faixas, columns=["campo", "faixas", "bits"]).astype({"faixas": int, "bits": int})
    display(df_faixas); registrar_tabela(_sec, "Faixas por campo", df_faixas)

# --- alertas ---
_n, _t = _conv["regras_lidas"], _conv["entradas_ternarias"]
if _n is None or _t is None:
    registrar_alerta(_sec, "Parse", "não consegui parsear regras/entradas ternárias do --report (formato mudou?)"); _st = "AVISO"
else:
    if _t > LIMITE_TERNARIAS_DETECT:
        registrar_alerta(_sec, "Tabela detect", f"{_t} entradas ternárias > {LIMITE_TERNARIAS_DETECT} (size da detect) — o p4c vai rejeitar", "FALHA"); _st = "FALHA"
    elif _t > FATOR_AVISO_TERNARIAS * _n:
        registrar_alerta(_sec, "Expansão", f"{_t} entradas ternárias > {FATOR_AVISO_TERNARIAS}×{_n} regras — alguma regra expandiu demais"); _st = "AVISO"

# --- artefatos regenerados nesta execução? ---
for _nome in (f"{PROG}.p4", "setup_rules.py"):
    _p = BUILD_DIR / _nome
    if not _p.is_file():
        registrar_alerta(_sec, "Artefato", f"{_p} não existe", "FALHA"); _st = "FALHA"; continue
    if datetime.fromtimestamp(_p.stat().st_mtime) < S.inicio:
        registrar_alerta(_sec, "Artefato", f"{_p.name} NÃO foi regenerado nesta sessão (mtime anterior ao início)", "FALHA"); _st = "FALHA"
    else:
        import shutil; shutil.copy2(_p, S.dir / _nome); print(f"copiado {_p.name} → {S.dir.name}/")
marcar(_sec, _st, _t0)

$ python3 rules2p4.py /home/lucas/Documentos/Mestrado/Goose/Conversor/rules_v1.py -o build --prog goose_ids --report
regras lidas ......... 20
campos ativos ........ 9
entradas ternárias ... 76
classes de ataque .... 8

build/goose_ids.p4
build/setup_rules.py

--- faixas por campo ---
SqNum                    6 faixas, 3 bits
StNum                    7 faixas, 3 bits
cbStatus                 2 faixas, 1 bits
delay                    2 faixas, 1 bits
sqDiff                   2 faixas, 1 bits
stDiff                   6 faixas, 3 bits
tDiff                    4 faixas, 2 bits
timeFromLastChange       4 faixas, 2 bits
timestampDiff            6 faixas, 3 bits

--- ataques ---
   1  grayhole
   2  high_StNum
   3  injection
   4  inverse_replay
   5  masquerade_fake_fault
   6  masquerade_fake_normal
   7  poisoned_high_rate
   8  random_replay
[rc=0, 0.1s]


,regras_lidas,campos_ativos,entradas_ternarias,classes_ataque
0,20,9,76,8


,campo,faixas,bits
0,SqNum,6,3
1,StNum,7,3
2,cbStatus,2,1
3,delay,2,1
4,sqDiff,2,1
5,stDiff,6,3
6,tDiff,4,2
7,timeFromLastChange,4,2
8,timestampDiff,6,3


copiado goose_ids.p4 → 20260915_171757_Secao_de_teste_do_modelo_de_regras_gerado_pela_gpt_oss/
copiado setup_rules.py → 20260915_171757_Secao_de_teste_do_modelo_de_regras_gerado_pela_gpt_oss/

✅ Seção 2: OK


## Seção 3 — Validação (`validate.py -n 100000`)

Critério: `divergência detecção` **e** `divergência classe` iguais a zero. Caso contrário a sessão é marcada **FALHA** e as seções seguintes bloqueiam (sobrescreva com `FORCAR_CONTINUACAO = True`, conscientemente).

In [10]:
# ----------------------------- PARÂMETROS ----------------------------
N_PACOTES_VALIDACAO = 100000
# ---------------------------------------------------------------------
exigir_etapas("1", "2"); _t0 = time.time(); _sec = "3"; _st = "OK"; S = SESSION

_r = sh(f"python3 validate.py {shlex.quote(str(S.regras))} -n {N_PACOTES_VALIDACAO}",
        _sec, "validate.py", cwd=CONVERSOR, timeout=1800)
salvar_saida("03_validacao.txt", _r.stdout + ("\n[stderr]\n" + _r.stderr if _r.stderr.strip() else ""))

def _num(rotulo, txt):
    m = re.search(rotulo + r"\s*\.*\s*(\d+)", txt)
    return int(m.group(1)) if m else None
_val = {
    "pacotes_testados":       _num(r"pacotes testados", _r.stdout),
    "deteccoes_coincidentes": _num(r"detec[cç][oõ]es coincidentes", _r.stdout),
    "divergencia_deteccao":   _num(r"diverg[eê]ncia detec[cç][aã]o", _r.stdout),
    "divergencia_classe":     _num(r"diverg[eê]ncia classe", _r.stdout),
}
S.dados["validacao"] = _val
df_val = pd.DataFrame([_val]); display(df_val); registrar_tabela(_sec, "Validação", df_val)

if not _r.ok or _val["divergencia_deteccao"] is None or _val["divergencia_classe"] is None:
    registrar_alerta(_sec, "Validação", "validate.py falhou ou saída não reconhecida", "FALHA"); _st = "FALHA"
elif _val["divergencia_deteccao"] != 0 or _val["divergencia_classe"] != 0:
    registrar_alerta(_sec, "Divergência",
        f"detecção={_val['divergencia_deteccao']} classe={_val['divergencia_classe']} ≠ 0 — a tradução não preserva a semântica. "
        "Causa provável: limiar com mais casas decimais do que a escala do campo (aumente scale em field_model.py e revalide).", "FALHA")
    _st = "FALHA"
marcar(_sec, _st, _t0)

$ python3 validate.py /home/lucas/Documentos/Mestrado/Goose/Conversor/rules_v1.py -n 100000
pacotes testados ......... 100000
detecções coincidentes ... 67885
divergência detecção ..... 1016
divergência classe ....... 0

regras com divergência:
    953  rule_masquerade_fake_fault_low_stnum_cbstatus
     35  rule_random_replay_timestamp_time
     34  rule_inverse_replay_low_sqnum_and_long_time
[rc=1, 16.4s]


,pacotes_testados,deteccoes_coincidentes,divergencia_deteccao,divergencia_classe
0,100000,67885,1016,0


❌ validate.py falhou ou saída não reconhecida

❌ Seção 3: FALHA


## Seção 4 — Compilação P4

Copia o `.p4` para `pkgsrc/p4-examples/p4_16_programs/goose_ids/` e executa o `p4c` de referência. Código de retorno zero **não basta**: `bf-rt.json`, `pipe/context.json`, `pipe/tofino.bin` e um `goose_ids.conf` JSON válido são obrigatórios.

O `p4c` direto não gera o `.conf`. Se ele não existir, a célula avisa e — **somente** se `CRIAR_CONF_DE_TNA_COUNTER = True` — deriva o `.conf` de `tna_counter.conf` via `sed` (método documentado no guia); alternativamente use `./p4_build.sh`.

In [7]:
# ----------------------------- PARÂMETROS ----------------------------
CRIAR_CONF_DE_TNA_COUNTER = True     # derivar goose_ids.conf de tna_counter.conf se não existir
TIMEOUT_P4C_S             = 1800
# ---------------------------------------------------------------------
exigir_etapas("1", "2", "3"); _t0 = time.time(); _sec = "4"; _st = "OK"; S = SESSION

sh(f"mkdir -p {shlex.quote(str(P4_DST))} && cp {shlex.quote(str(BUILD_DIR / (PROG + '.p4')))} {shlex.quote(str(P4_DST))}/",
   _sec, "copiar .p4 para pkgsrc")
_cmd_p4c = (f"p4c --target tofino --arch tna --program-name {PROG} "
            f"--bf-rt-schema {shlex.quote(str(TOFINOPD / 'bf-rt.json'))} "
            f"-o {shlex.quote(str(TOFINOPD))} {shlex.quote(str(P4_DST / (PROG + '.p4')))}")
_r = sh(_cmd_p4c, _sec, "p4c", timeout=TIMEOUT_P4C_S)
salvar_saida("04_p4c.txt", _r.stdout + "\n[stderr]\n" + _r.stderr)
if not _r.ok:
    _txt = _r.stdout + _r.stderr
    if "table placement failed" in _txt or "not enough stages" in _txt:
        registrar_alerta(_sec, "Recursos", "table placement failed / not enough stages → reduza regras ou consolide limiares próximos.", "FALHA")
    elif "PHV nibbles" in _txt:
        registrar_alerta(_sec, "Range match", "range match aceita no máximo 16 bits (4 nibbles) → verifique larguras em field_model.py.", "FALHA")
    _st = "FALHA"

# artefatos obrigatórios
_falta = [f for f in ("bf-rt.json", "pipe/context.json", "pipe/tofino.bin") if not (TOFINOPD / f).is_file()]
if _falta:
    registrar_alerta(_sec, "Artefatos", "ausentes: " + ", ".join(_falta) + " — compilação falhou mesmo que o rc seja 0", "FALHA"); _st = "FALHA"
else:
    print("✅ bf-rt.json, pipe/context.json, pipe/tofino.bin presentes")

# .conf
if not CONF.is_file():
    _tna = CONF.with_name("tna_counter.conf")
    if CRIAR_CONF_DE_TNA_COUNTER and _tna.is_file():
        print(f"⚠️ {CONF.name} não existia — derivando de tna_counter.conf (CRIAR_CONF_DE_TNA_COUNTER=True)")
        sh(f"sed 's/tna_counter/{PROG}/g' {shlex.quote(str(_tna))} > {shlex.quote(str(CONF))}", _sec, "gerar .conf via sed")
    else:
        registrar_alerta(_sec, ".conf", f"{CONF} não existe (o p4c direto não o gera). Opções: CRIAR_CONF_DE_TNA_COUNTER=True "
                         f"ou compilar com `cd {SDE_HOME} && ./p4_build.sh pkgsrc/p4-examples/p4_16_programs/{PROG}/{PROG}.p4`.", "FALHA")
        _st = "FALHA"
if CONF.is_file():
    try:
        json.loads(CONF.read_text()); print(f"✅ {CONF.name} é JSON válido")
        registrar(_sec, ".conf válido", f"python3 -m json.tool {CONF}", stdout="JSON valido", tipo="info")
    except json.JSONDecodeError as e:
        registrar_alerta(_sec, ".conf", f"{CONF} inválido: {e}", "FALHA"); _st = "FALHA"

sh(f"ls -la {shlex.quote(str(TOFINOPD))}/ {shlex.quote(str(TOFINOPD))}/pipe/ 2>&1", _sec, "ls tofinopd")
S.dados["compilacao"] = {"artefatos_faltando": _falta, "conf": str(CONF), "p4c_rc": _r.returncode}
marcar(_sec, _st, _t0)

RuntimeError: ❌ Seção 3 está em FALHA. Corrija ou defina FORCAR_CONTINUACAO = True.

## Seção 5 — Processos de longa duração: `tofino-model` e `bf_switchd`

Ambos em background (`Popen`, `start_new_session=True`, stdout em log no diretório da sessão). Ordem: modelo → espera → switchd. Pronto quando a porta 50052 aceita conexão **ou** o log mostra `bfruntime gRPC server started`.

`run_tofino_model.sh`/`run_switchd.sh` chamam `sudo` internamente; sem tty não haveria como digitar a senha, por isso `LANCAR_COM_SUDO = True` lança-os via `sudo -S` (senha da Seção 1). O `stdin` do switchd fica aberto (é o `bfshell` embutido).

In [ ]:
# ----------------------------- PARÂMETROS ----------------------------
TIMEOUT_PRONTO_S   = 180
ESPERA_APOS_MODELO = 8        # segundos entre subir o modelo e o switchd
PORTA_GRPC         = 50052
LANCAR_COM_SUDO    = True
PERGUNTAR_ANTES_DE_ENCERRAR_ANTERIORES = True
# ---------------------------------------------------------------------
exigir_etapas("1", "4"); _t0 = time.time(); _sec = "5"; _st = "OK"; S = SESSION
PROCS = globals().get("PROCS", {})   # 'model'/'switchd' -> dict(popen, log, pgid)

def _porta_aberta(porta, host="127.0.0.1"):
    with socket.socket() as sk:
        sk.settimeout(0.5)
        return sk.connect_ex((host, porta)) == 0

def _tail(path, n=8):
    try:
        return "\n".join(Path(path).read_text(errors="replace").splitlines()[-n:])
    except Exception:
        return ""

def encerrar_grupo(nome, pgid, timeouts=(5, 5, 2)):
    """SIGINT → SIGTERM → SIGKILL no grupo de processos (via sudo, pois os filhos podem ser root)."""
    for sig, t in zip(("INT", "TERM", "KILL"), timeouts):
        _r = sh(f"kill -{sig} -- -{pgid} 2>/dev/null; sleep {t}; pgrep -g {pgid} >/dev/null && echo VIVO || echo MORTO",
                "9", f"encerrar {nome} ({sig})", sudo=True, timeout=t + 10, mostrar=False)
        if "MORTO" in _r.stdout:
            print(f"  {nome}: encerrado após SIG{sig}"); return True
    print(f"  {nome}: ⚠️ ainda vivo após SIGKILL"); return False

def encerrar_todos():
    for nome in ("switchd", "model"):
        info = PROCS.pop(nome, None)
        if info:
            encerrar_grupo(nome, info["pgid"])
            try: info["popen"].stdin and info["popen"].stdin.close()
            except Exception: pass
_atexit_registrado = globals().get("_atexit_registrado", False)
if not _atexit_registrado:
    atexit.register(lambda: (globals().get("BFSHELL") and BFSHELL.close(), encerrar_todos()))
    _atexit_registrado = True

def status_processos():
    for nome, info in PROCS.items():
        vivo = info["popen"].poll() is None
        print(f"[{nome}] pgid={info['pgid']} {'VIVO' if vivo else 'MORTO rc=%s' % info['popen'].returncode} log={info['log']}")
        print("   " + _tail(info["log"], 5).replace("\n", "\n   "))

# instâncias anteriores
_r = sh("pgrep -fa 'tofino-model|bf_switchd' | grep -v pgrep || true", _sec, "instâncias anteriores", mostrar=False)
if _r.stdout.strip():
    print("⚠️ processos já em execução:\n" + _r.stdout)
    _resp = input("Encerrar essas instâncias? [s/N] ").strip().lower() if PERGUNTAR_ANTES_DE_ENCERRAR_ANTERIORES else "s"
    if _resp == "s":
        encerrar_todos()
        sh("pkill -INT -f 'tofino-model' ; pkill -INT -f 'bf_switchd' ; sleep 3; pkill -KILL -f 'tofino-model|bf_switchd' ; true",
           _sec, "pkill instâncias anteriores", sudo=True, timeout=30)
    else:
        marcar(_sec, "FALHA", _t0); raise RuntimeError("❌ instâncias anteriores mantidas — encerre-as antes de continuar.")

def lancar(nome, script_args, log_nome):
    log_path = S.dir / log_nome
    logf = open(log_path, "ab")
    cmd = f"cd {shlex.quote(str(SDE_HOME))} && exec {script_args}"
    if LANCAR_COM_SUDO:
        argv = ["sudo", "-S", "-p", "", "-E", "env", f"PATH={os.environ.get('PATH','')}", "bash", "-c", cmd]
    else:
        argv = ["bash", "-c", cmd]
    p = subprocess.Popen(argv, stdin=subprocess.PIPE, stdout=logf, stderr=subprocess.STDOUT,
                         start_new_session=True, env=os.environ.copy())
    if LANCAR_COM_SUDO:
        p.stdin.write((_SECRETS["sudo"] + "\n").encode()); p.stdin.flush()
    PROCS[nome] = {"popen": p, "log": str(log_path), "pgid": os.getpgid(p.pid)}
    S.pids[nome] = os.getpgid(p.pid)
    registrar(_sec, f"lançar {nome}", script_args, stdout=f"pgid={os.getpgid(p.pid)} log={log_path}", tipo="info")
    print(f"▶ {nome} lançado (pgid={os.getpgid(p.pid)}) → {log_path.name}")
    return p

if "model" not in PROCS or PROCS["model"]["popen"].poll() is not None:
    lancar("model", f"./run_tofino_model.sh -p {PROG} --arch tofino", "tofino_model.log")
    for _i in range(ESPERA_APOS_MODELO):
        time.sleep(1)
        if PROCS["model"]["popen"].poll() is not None:
            marcar(_sec, "FALHA", _t0); print(_tail(PROCS["model"]["log"], 30))
            raise RuntimeError("❌ tofino-model terminou prematuramente — veja tofino_model.log acima (conf ausente? porta ocupada?)")
else:
    print("model já em execução")

if "switchd" not in PROCS or PROCS["switchd"]["popen"].poll() is not None:
    lancar("switchd", f"./run_switchd.sh -p {PROG} --arch tofino", "switchd.log")
else:
    print("switchd já em execução")

# espera ativa
_ini = time.time(); _pronto = False
while time.time() - _ini < TIMEOUT_PRONTO_S:
    _log = Path(PROCS["switchd"]["log"]).read_text(errors="replace") if Path(PROCS["switchd"]["log"]).exists() else ""
    if _porta_aberta(PORTA_GRPC) or "bfruntime gRPC server started" in _log:
        _pronto = True; break
    if PROCS["switchd"]["popen"].poll() is not None:
        break
    _dt = int(time.time() - _ini)
    if _dt % 10 == 0:
        print(f"  … {_dt}s | switchd: {_tail(PROCS['switchd']['log'], 1)[:110]}")
    time.sleep(1)

_dt = time.time() - _ini
if _pronto:
    print(f"✅ bf_switchd pronto em {_dt:.0f}s (porta {PORTA_GRPC})")
    registrar(_sec, "switchd pronto", None, stdout=f"{_dt:.0f}s\n" + _tail(PROCS["switchd"]["log"], 10), tipo="info")
else:
    registrar_alerta(_sec, "switchd", f"não ficou pronto em {TIMEOUT_PRONTO_S}s. Últimas linhas:\n" + _tail(PROCS["switchd"]["log"], 20), "FALHA")
    print("model tail:\n" + _tail(PROCS["model"]["log"], 10)); _st = "FALHA"
status_processos()
marcar(_sec, _st, _t0)

**Célula auxiliar de status** (reexecute quando quiser):

In [ ]:
exigir_sessao(); status_processos()

## Seção 6 — Sessão `bfshell`/`bfrt_python` persistente (núcleo)

`pexpect` abre um pty real (equivalente ao `script -c` do fluxo manual), entra em `bfrt_python`, detecta o prompt **dinamicamente** e carrega `setup_rules.py` **uma única vez**. A sessão permanece aberta para a Seção 7 alternar injeção e `hits()`. Toda a interação é espelhada em `bfshell_session.log`.

In [ ]:
# ----------------------------- PARÂMETROS ----------------------------
TIMEOUT_BFSHELL_S = 60      # espera pelo prompt bfshell>
TIMEOUT_CMD_S     = 120     # padrão por comando
COMANDOS_BFRT     = [       # comandos ad hoc executados após a carga (edite à vontade)
    "d.info(return_info=False)",
]
# ---------------------------------------------------------------------
exigir_etapas("1", "5"); _t0 = time.time(); _sec = "6"; _st = "OK"; S = SESSION
if pexpect is None:
    raise RuntimeError("❌ pexpect não instalado: pip install pexpect")

class BfShell:
    """Sessão persistente ./run_bfshell.sh → bfrt_python. Uma instância por sessão."""
    SENTINELA = "__NB_SENTINELA__"
    def __init__(self, cwd, log_path, secao="6"):
        self.cwd, self.log_path, self.secao = Path(cwd), Path(log_path), secao
        self.child = None; self.prompt = None; self._logf = None
    # --- ciclo de vida ---
    def start(self):
        self._logf = open(self.log_path, "a", encoding="utf-8")
        self.child = pexpect.spawn("./run_bfshell.sh", cwd=str(self.cwd), env=os.environ.copy(),
                                   encoding="utf-8", codec_errors="replace", timeout=TIMEOUT_BFSHELL_S,
                                   dimensions=(50, 400))
        self.child.logfile_read = self._logf
        self.child.expect(r"bfshell>\s*", timeout=TIMEOUT_BFSHELL_S)
        self.child.sendline("bfrt_python")
        # detectar o prompt real do bfrt_python: imprime uma sentinela e lê o que vem depois dela
        time.sleep(1.5)
        self.child.sendline(f'print("{self.SENTINELA}")')
        self.child.expect(re.escape(self.SENTINELA) + r"\r?\n", timeout=TIMEOUT_BFSHELL_S)   # eco da linha
        self.child.expect(re.escape(self.SENTINELA) + r"\r?\n", timeout=TIMEOUT_BFSHELL_S)   # saída do print
        try:
            self.child.expect(pexpect.TIMEOUT, timeout=1.5)
        except Exception:
            pass
        linhas = [l for l in limpar_ansi(self.child.before).splitlines() if l.strip()]
        self.prompt = linhas[-1].rstrip() if linhas else "bfrt>"
        if not self.prompt.endswith(">"):
            print(f"⚠️ prompt detectado não termina em '>': {self.prompt!r} — confira em bfshell_session.log")
        registrar(self.secao, "bfrt_python iniciado", "./run_bfshell.sh → bfrt_python", stdout=f"prompt detectado: {self.prompt!r}", tipo="info")
        print(f"✅ bfrt_python pronto (prompt {self.prompt!r})")
        return self
    def _esperar_prompt(self, timeout):
        # o prompt pode vir com ou sem espaço final; usa o texto sem espaço final
        self.child.expect_exact(self.prompt.rstrip(), timeout=timeout)
    def run(self, cmd, timeout=None, titulo=None, mostrar=True):
        timeout = timeout or TIMEOUT_CMD_S
        ini = datetime.now()
        self.child.sendline(cmd)
        try:
            self._esperar_prompt(timeout); status = "OK"
            saida = limpar_ansi(self.child.before)
        except pexpect.TIMEOUT:
            saida = limpar_ansi(self.child.before or ""); status = "FALHA"
            saida += f"\n[TIMEOUT {timeout}s aguardando o prompt {self.prompt!r}]"
        linhas = saida.splitlines()
        if linhas and linhas[0].strip() == cmd.strip():
            linhas = linhas[1:]                                   # remove o eco
        saida = "\n".join(linhas).strip("\n")
        if re.search(r"Traceback|Error", saida) and status == "OK":
            status = "AVISO"
        registrar(self.secao, titulo or f"bfrt: {cmd[:60]}", cmd, stdout=saida, returncode=0 if status == "OK" else 1,
                  inicio=ini, fim=datetime.now(), status=status, tipo="bfrt")
        if mostrar:
            print(f"{self.prompt} {cmd}\n{saida}")
        return saida
    def run_block(self, code, timeout=None, titulo=None, mostrar=True):
        """Código multilinha: via exec(repr) em uma linha quando cabe no pty; senão linha a linha."""
        linha = "exec(" + repr(code) + ")"
        if len(linha) < 3500:
            return self.run(linha, timeout, titulo or "bfrt: bloco", mostrar)
        saidas = []
        for l in code.splitlines():
            self.child.sendline(l); time.sleep(0.05)
        self.child.sendline("")                                    # fecha blocos pendentes
        return self.run("", timeout, titulo or "bfrt: bloco (linha a linha)", mostrar)
    def close(self):
        if self.child is None:
            return
        try:
            if self.child.isalive():
                self.child.sendline("exit()"); time.sleep(0.5)
                self.child.sendline("exit"); time.sleep(0.5)
                self.child.close(force=True)
        except Exception:
            pass
        try: self._logf and self._logf.close()
        except Exception: pass
        self.child = None
        print("bfshell encerrado")

# uma sessão por experimento: fecha a anterior se existir
if globals().get("BFSHELL") is not None:
    try: BFSHELL.close()
    except Exception: pass
BFSHELL = BfShell(SDE_HOME, S.dir / "bfshell_session.log").start()

# carga das regras — UMA vez; não usar também run_bfshell.sh -b (duplicaria entradas)
_setup_rules = BUILD_DIR / "setup_rules.py"
assert _setup_rules.is_file(), f"{_setup_rules} não existe (Seção 2)"
BFSHELL.run(f"d = bfrt.{PROG}.pipe.Ingress.detect", titulo="alias d = detect")
_out = BFSHELL.run(f"exec(open({str(_setup_rules)!r}).read())", timeout=300, titulo="exec(setup_rules.py)")
salvar_saida("06_setup_rules_out.txt", _out)

# parse das contagens impressas pelo setup_rules.py
_bands = {m.group(1): int(m.group(2)) for m in re.finditer(r"(tbl_band_\w+)\s*:?\s*(\d+)\s+faixas", _out)}
_m_det = re.search(r"detect\s*:?\s*(\d+)\s+entradas", _out)
_n_detect = int(_m_det.group(1)) if _m_det else None
S.dados["controle"] = {"bands": _bands, "detect": _n_detect, "ok": "OK - regras carregadas" in _out}
if "Traceback" in _out or "unexpected keyword argument" in _out:
    registrar_alerta(_sec, "API BF-Runtime",
        "setup_rules.py falhou. Se for 'unexpected keyword argument', os nomes dos parâmetros de range/ternary variam entre versões do SDE: "
        f"inspecione com bfrt.{PROG}.pipe.Ingress.tbl_band_SqNum.info(return_info=False) e ajuste bfrt_emitter.py.", "FALHA"); _st = "FALHA"
elif not S.dados["controle"]["ok"]:
    registrar_alerta(_sec, "Carga", "não vi 'OK - regras carregadas' na saída — confira 06_setup_rules_out.txt"); _st = "AVISO"
_esp = S.dados.get("conversao", {}).get("entradas_ternarias")
if _n_detect is not None and _esp is not None:
    if _n_detect == _esp:
        print(f"✅ detect: {_n_detect} entradas carregadas = {_esp} entradas ternárias da conversão")
    else:
        registrar_alerta(_sec, "detect", f"detect carregou {_n_detect} entradas, conversão previa {_esp}"); _st = "AVISO"
if _bands:
    df_b = pd.DataFrame(sorted(_bands.items()), columns=["tabela", "faixas"]); display(df_b); registrar_tabela(_sec, "Faixas carregadas", df_b)

# comandos ad hoc
for _c in COMANDOS_BFRT:
    BFSHELL.run(_c, titulo=f"ad hoc: {_c[:50]}")
marcar(_sec, _st, _t0)

**Célula livre** — edite `COMANDOS_BFRT` e reexecute quantas vezes quiser (a sessão continua aberta):

In [ ]:
COMANDOS_BFRT = [
    "hits()",
    # "d.operation_counter_sync(); d.dump(from_hw=True)",
    # f"bfrt.{PROG}.pipe.Ingress.tbl_band_SqNum.dump()",
]
exigir_sessao()
for _c in COMANDOS_BFRT:
    BFSHELL.run(_c, titulo=f"ad hoc: {_c[:50]}")

## Seção 7 — Laço de experimento: injeção + `hits()`

Gera o PCAP com `gen_test_traffic.py`, parseia o veredito esperado e roda `N_RODADAS` de: `hits()` → `tcpreplay` → pausa → `hits()` → **delta por classe** (os contadores são cumulativos). Compara com o esperado do gerador e salva `hits_rodadas.csv` / `resumo_rodadas.csv`.

**Formato assumido de `hits()`** (observado no guia; se mudar, a saída bruta é preservada e a célula avisa):
```
   3 pkts  grayhole
   2 pkts  high_StNum
---
8 classes com trafego, 17 pacotes no total
```
Classes sem tráfego não aparecem (tratadas como 0). `hits()` agrega por classe; para contadores **por entrada** use `PER_ENTRADA_DUMP = True` (`d.dump(from_hw=True)`, saída bruta anexada ao relatório).

In [ ]:
# ----------------------------- PARÂMETROS ----------------------------
N_RODADAS          = 3
IFACE              = "veth0"
ESPERA_POS_REPLAY_S = 2
INCLUIR_UNTAGGED   = False        # também gerar/injetar test_untagged.pcap (--untagged)
PER_ENTRADA_DUMP   = False        # anexar d.dump(from_hw=True) bruto a cada rodada
TIMEOUT_TCPREPLAY_S = 300
# ---------------------------------------------------------------------
exigir_etapas("1", "5", "6"); _t0 = time.time(); _sec = "7"; _st = "OK"; S = SESSION
if BFSHELL is None or BFSHELL.child is None or not BFSHELL.child.isalive():
    raise RuntimeError("❌ sessão bfrt_python não está viva — reexecute a Seção 6")

# --- 1. gerar PCAPs ---
def gerar_pcap(nome, untagged=False):
    pcap = S.dir / nome
    _r = sh(f"python3 gen_test_traffic.py {shlex.quote(str(S.regras))} -o {shlex.quote(str(pcap))}{' --untagged' if untagged else ''}",
            _sec, f"gen_test_traffic.py → {nome}", cwd=CONVERSOR, timeout=300)
    salvar_saida(f"07_gen_{nome}.txt", _r.stdout + _r.stderr)
    if not _r.ok or not pcap.is_file():
        raise RuntimeError(f"❌ gen_test_traffic.py falhou para {nome}")
    linhas = re.findall(r"^\s*(\d+)\s+(\S+)\s+(\S+)\s+(.*?)\s*$", _r.stdout, re.M)
    tab = pd.DataFrame([(int(i), c, e, r) for i, c, e, r in linhas if c != "caso" and e != "->"],
                       columns=["n", "caso", "esperado", "regras"])
    m = re.search(r"(\d+)\s+pacotes\s*->.*?\((\d+)\s+devem casar em detect,\s*(\d+)\s+normais\)", _r.stdout)
    tot = dict(pacotes=int(m.group(1)), ataque=int(m.group(2)), normais=int(m.group(3))) if m else \
          dict(pacotes=len(tab), ataque=int((tab.esperado != "NORMAL").sum()), normais=int((tab.esperado == "NORMAL").sum()))
    esperado_por_classe = tab[tab.esperado != "NORMAL"].groupby("esperado").size().to_dict()
    return dict(nome=nome, pcap=pcap, tabela=tab, totais=tot, por_classe=esperado_por_classe)

PCAPS = [gerar_pcap("test_goose.pcap")]
if INCLUIR_UNTAGGED:
    PCAPS.append(gerar_pcap("test_untagged.pcap", untagged=True))
for _p in PCAPS:
    print(f"\n{_p['nome']}: {_p['totais']}  esperado/classe={_p['por_classe']}")
    registrar_tabela(_sec, f"Veredito esperado — {_p['nome']}", _p["tabela"])
S.dados["pcaps"] = {p["nome"]: dict(totais=p["totais"], por_classe=p["por_classe"]) for p in PCAPS}

# --- 2. parser de hits() ---
_RE_HIT   = re.compile(r"^\s*(\d+)\s+pkts?\s+(\S+)\s*$", re.M)
_RE_TOTAL = re.compile(r"(\d+)\s+classes?\s+com\s+tr[aá]fego,\s*(\d+)\s+pacotes?\s+no\s+total")
def ler_hits(titulo):
    bruto = BFSHELL.run("hits()", titulo=titulo, mostrar=False)
    por_classe = {cl: int(n) for n, cl in _RE_HIT.findall(bruto)}
    m = _RE_TOTAL.search(bruto)
    total = int(m.group(2)) if m else None
    reconhecido = bool(por_classe) or (total == 0) or ("0 classes" in bruto) or (m is not None)
    if not reconhecido:
        registrar_alerta(_sec, "hits() formato", f"saída de hits() não reconhecida (bruto preservado no log):\n{bruto[:400]}")
    return dict(bruto=bruto, por_classe=por_classe, total=total, reconhecido=reconhecido)

# --- 3. rodada ---
_RE_TR = re.compile(r"Actual:\s*(\d+)\s+packets.*?\n.*?Successful packets:\s*(\d+).*?Failed packets:\s*(\d+)", re.S)
def rodada(i, pcap_info, iface=IFACE):
    antes = ler_hits(f"rodada {i} — hits() antes")
    r = sh(f"tcpreplay -i {iface} {shlex.quote(str(pcap_info['pcap']))}", _sec, f"rodada {i} — tcpreplay {pcap_info['nome']}",
           sudo=True, timeout=TIMEOUT_TCPREPLAY_S, mostrar=False)
    m = _RE_TR.search(r.stdout + r.stderr)
    tr = dict(enviados=int(m.group(1)), ok=int(m.group(2)), falhos=int(m.group(3))) if m else dict(enviados=None, ok=None, falhos=None)
    time.sleep(ESPERA_POS_REPLAY_S)
    depois = ler_hits(f"rodada {i} — hits() depois")
    if PER_ENTRADA_DUMP:
        dump = BFSHELL.run("d.operation_counter_sync(); d.dump(from_hw=True)", timeout=300, titulo=f"rodada {i} — dump detect", mostrar=False)
        salvar_saida(f"07_dump_rodada{i}.txt", dump)
    classes = sorted(set(antes["por_classe"]) | set(depois["por_classe"]) | set(pcap_info["por_classe"]))
    linhas = []
    for cl in classes:
        a, d = antes["por_classe"].get(cl, 0), depois["por_classe"].get(cl, 0)
        linhas.append(dict(rodada=i, pcap=pcap_info["nome"], classe=cl, hits_antes=a, hits_depois=d, delta=d - a,
                           esperado=pcap_info["por_classe"].get(cl, 0)))
    df = pd.DataFrame(linhas)
    tot_antes = antes["total"] if antes["total"] is not None else sum(antes["por_classe"].values())
    tot_depois = depois["total"] if depois["total"] is not None else sum(depois["por_classe"].values())
    delta_total = tot_depois - tot_antes
    esperado_ataque = pcap_info["totais"]["ataque"]
    divergente = (not df.empty and (df.delta != df.esperado).any()) or delta_total != esperado_ataque
    resumo = dict(rodada=i, pcap=pcap_info["nome"], tcpreplay_enviados=tr["enviados"], tcpreplay_ok=tr["ok"],
                  flag_attack_delta=delta_total, ataques_esperados=esperado_ataque,
                  no_attack_esperado=pcap_info["totais"]["normais"],
                  classes_ok=int((df.delta == df.esperado).sum()) if not df.empty else 0,
                  classes_total=len(df), status="DIVERGENTE" if divergente else "OK",
                  hits_reconhecido=antes["reconhecido"] and depois["reconhecido"])
    print(f"rodada {i} [{pcap_info['nome']}] tcpreplay={tr} | flag_attack Δ={delta_total} (esperado {esperado_ataque}) → {resumo['status']}")
    return df, resumo

# --- 4. laço ---
RESULTADOS, RESUMOS = [], []
_i = 0
for _p in PCAPS:
    for _k in range(N_RODADAS):
        _i += 1
        _df, _res = rodada(_i, _p)
        RESULTADOS.append(_df); RESUMOS.append(_res)

df_hits   = pd.concat(RESULTADOS, ignore_index=True) if RESULTADOS else pd.DataFrame()
df_resumo = pd.DataFrame(RESUMOS)
df_hits.to_csv(S.dir / "hits_rodadas.csv", index=False)
df_resumo.to_csv(S.dir / "resumo_rodadas.csv", index=False)
display(df_resumo); registrar_tabela(_sec, "Resumo por rodada", df_resumo)
if not df_hits.empty:
    display(df_hits); registrar_tabela(_sec, "Hits por rodada × classe", df_hits)
S.dados["experimento"] = dict(rodadas=len(RESUMOS), divergentes=int((df_resumo.status == "DIVERGENTE").sum()) if not df_resumo.empty else 0)

# --- 5. diagnóstico automático se tudo zero ---
if not df_hits.empty and (df_hits.delta == 0).all():
    registrar_alerta(_sec, "Diagnóstico", "todos os deltas são zero — executando diagnóstico automático", "FALHA"); _st = "FALHA"
    _d = sh(f"tcpdump -r {shlex.quote(str(PCAPS[0]['pcap']))} -xx -c 1 2>&1 | head -6", _sec, "diag: EtherType no PCAP")
    if "88b8" not in _d.stdout.lower():
        print("   → EtherType 0x88B8 NÃO visível no primeiro pacote: o parser GOOSE não casa (confira o log do modelo: só 'ethernet' válido?)")
    _b = next(iter(S.dados.get("controle", {}).get("bands", {})), "tbl_band_SqNum")
    BFSHELL.run(f"bfrt.{PROG}.pipe.Ingress.{_b}.dump()", timeout=120, titulo=f"diag: dump {_b}")
    print(f"   → se a tabela acima estiver vazia, as faixas não foram populadas (Seção 6).")
    print(f"   → interface: o modelo escuta em um lado do par; tente IFACE = 'veth1'.")
    print("   → log do modelo:\n" + _tail(PROCS["model"]["log"], 15))
elif not df_resumo.empty and (df_resumo.status == "DIVERGENTE").any():
    registrar_alerta(_sec, "Divergência", "hits divergem do esperado em alguma rodada. Com validate.py em 0, verifique se GOOSE_FIELDS em "
                     "gen_test_traffic.py está na mesma ordem de goose_feat_h no goose_ids.p4."); _st = "AVISO"
marcar(_sec, _st, _t0)

## Seção 8 — Relatório consolidado

Gera `relatorio.md` (e `relatorio.html` se o módulo `markdown` estiver instalado) no diretório da sessão a partir de `REPORT`, com status por seção, métricas, tabelas e a saída integral de cada bloco em `<details>` (truncada no corpo quando muito longa; íntegra nos logs).

In [ ]:
exigir_sessao(); _t0 = time.time(); _sec = "8"; S = SESSION

def _trunc(txt, lim=TRUNCAR_RELATORIO_CHARS):
    txt = txt or ""
    if len(txt) <= lim:
        return txt
    h, t = lim * 2 // 3, lim // 3
    return txt[:h] + f"\n\n[... truncado: {len(txt) - lim} caracteres omitidos — íntegra nos logs da sessão ...]\n\n" + txt[-t:]

def _md_escape(s):
    return str(s).replace("|", "\\|")

_p4c_ver = next((e["stdout"].strip().splitlines()[0] for e in REPORT if e["titulo"] == "p4c --version" and e["stdout"].strip()), "?")
L = []
L += [f"# Relatório — {S.nome}", ""]
L += [f"- **Arquivo de regras:** `{S.regras}`  ", f"- **SHA-256:** `{sha256_arquivo(S.regras)}`  ",
      f"- **Início:** {S.inicio:%Y-%m-%d %H:%M:%S}  — **Geração:** {datetime.now():%Y-%m-%d %H:%M:%S}  ",
      f"- **Host:** {platform.node()} ({platform.platform()})  ", f"- **Python:** {sys.version.split()[0]}  — **p4c:** {_p4c_ver}  ",
      f"- **SDE:** `{os.environ.get('SDE','')}`  ", f"- **Diretório da sessão:** `{S.dir}`", ""]

L += ["## Status por seção", "", "| Seção | Status | Duração (s) |", "|---|---|---|"]
_nomes = {"0": "Sessão", "1": "Ambiente", "2": "Conversão", "3": "Validação", "4": "Compilação", "5": "Processos", "6": "bfrt_python", "7": "Experimento", "8": "Relatório", "9": "Encerramento"}
for k in sorted(_nomes):
    st = S.status.get(k, "—"); ic = {"OK": "✅", "AVISO": "⚠️", "FALHA": "❌"}.get(st, "")
    L.append(f"| {k} — {_nomes[k]} | {ic} {st} | {S.duracao.get(k, 0):.1f} |")
L.append("")

def _dict_tab(titulo, d):
    if not d: return
    L.extend([f"## {titulo}", "", "| chave | valor |", "|---|---|"])
    for k, v in d.items():
        L.append(f"| {_md_escape(k)} | {_md_escape(v)} |")
    L.append("")
_c = dict(S.dados.get("conversao", {})); _c.pop("faixas", None); _c.pop("ataques", None)
_dict_tab("Métricas da conversão", _c)
_dict_tab("Validação", S.dados.get("validacao"))
_dict_tab("Compilação", S.dados.get("compilacao"))
_ctl = dict(S.dados.get("controle", {})); _bands = _ctl.pop("bands", {})
_dict_tab("Plano de controle (setup_rules.py)", _ctl)
if _bands: _dict_tab("Faixas carregadas por tabela", _bands)
for nome, info in S.dados.get("pcaps", {}).items():
    _dict_tab(f"PCAP {nome} — totais", info["totais"]); _dict_tab(f"PCAP {nome} — esperado por classe", info["por_classe"])
for e in REPORT:
    if e["tipo"] == "tabela" and e["secao"] == "7":
        L += [f"## {e['titulo']}", "", e["extra"].get("markdown", e["stdout"]), ""]

L += ["## Alertas", ""]
_al = [e for e in REPORT if e["tipo"] == "alerta"]
L += [f"- **[{e['secao']}] {e['status']}** — {e['titulo']}: {e['stdout'].splitlines()[0] if e['stdout'] else ''}" for e in _al] or ["(nenhum)"]
L.append("")

L += ["## Anexo — saída integral por bloco", ""]
for n, e in enumerate(REPORT, 1):
    ic = {"OK": "✅", "AVISO": "⚠️", "FALHA": "❌"}.get(e["status"], "")
    dur = f" · {(e['fim'] - e['inicio']).total_seconds():.1f}s" if e.get("inicio") and e.get("fim") else ""
    L.append(f"<details><summary>{n:03d} [{e['secao']}] {ic} {_md_escape(e['titulo'])}{dur}</summary>\n")
    if e["comando"]: L.append(f"```\n$ {e['comando']}\n```")
    corpo = e["stdout"] + (("\n[stderr]\n" + e["stderr"]) if e["stderr"].strip() else "")
    L.append("```\n" + _trunc(corpo).replace("```", "` ` `") + "\n```")
    L.append("\n</details>\n")
L += ["## Logs brutos", ""] + [f"- `{p.name}`" for p in sorted(S.dir.iterdir()) if p.suffix in (".log", ".txt", ".csv", ".pcap", ".p4", ".py")]

_md = _redigir("\n".join(L))
(S.dir / "relatorio.md").write_text(_md, encoding="utf-8")
print(f"✅ {S.dir / 'relatorio.md'} ({len(_md)} chars)")
try:
    import markdown as _mdlib
    _html = _mdlib.markdown(_md, extensions=["tables", "fenced_code"])
    (S.dir / "relatorio.html").write_text(
        "<!doctype html><meta charset='utf-8'><style>body{font-family:sans-serif;max-width:1100px;margin:auto}"
        "pre{background:#f4f4f4;padding:8px;overflow-x:auto}table{border-collapse:collapse}td,th{border:1px solid #999;padding:3px 8px}</style>"
        + _html, encoding="utf-8")
    print(f"✅ {S.dir / 'relatorio.html'}")
except ImportError:
    print("ℹ️ módulo 'markdown' ausente — só relatorio.md (pip install markdown)")
marcar(_sec, "OK", _t0)

## Seção 9 — Encerramento

Fecha o `bfrt_python`/`bfshell`, depois encerra `bf_switchd` e `tofino-model` pelo grupo de processos (SIGINT → SIGTERM → SIGKILL). Confere que não sobrou nada.

In [ ]:
exigir_sessao(); _t0 = time.time(); _sec = "9"; _st = "OK"; S = SESSION
if globals().get("BFSHELL") is not None:
    BFSHELL.close(); BFSHELL = None
encerrar_todos()
_r = sh("pgrep -fa 'tofino-model|bf_switchd|bfshell' | grep -v pgrep || echo NENHUM", _sec, "processos restantes")
if "NENHUM" not in _r.stdout:
    registrar_alerta(_sec, "Órfãos", "ainda há processos — tentando pkill -KILL", "AVISO"); _st = "AVISO"
    sh("pkill -KILL -f 'tofino-model|bf_switchd|bfshell' ; sleep 1; pgrep -fa 'tofino-model|bf_switchd|bfshell' | grep -v pgrep || echo NENHUM",
       _sec, "pkill -KILL", sudo=True)
marcar(_sec, _st, _t0)
print("\nSessão concluída. Para outra sessão com outro conjunto de regras: reexecute a Seção 0b e siga a partir da Seção 1 (a senha do sudo é reaproveitada).")

### Nova sessão sem reiniciar o kernel

Reexecute a **Seção 0b**, escolha outro `rules_*.py`, confirme, e rode da Seção 1 em diante. `REPORT` é zerado na confirmação; os processos anteriores já foram encerrados na Seção 9 (a Seção 5 detecta e pergunta se houver sobras).